[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ncsu-geoforall-lab/gis-584-uas-course/blob/gh-pages/course/topics/topic_5_advanced_analytics/assignments/fusion_analysis.ipynb)

# Assignment 5A: Fusion of UAS and lidar data

**Course:** GIS/MAE 584 Mapping and Analytics Using UAS
**Institution:** NC State University, Center for Geospatial Analytics
**Instructor:** Corey White

In this notebook we combine the 2013 airborne lidar surfaces of Mid Pines with the June 2015 UAS SfM DSM. We first compare the lidar DSM with the UAS DSMs from two SfM packages, as difference maps and along a profile on the road, to find systematic shifts and distortions. We then patch the least distorted part of the UAS DSM into the lidar DSM, once by a simple patch and once by a weighted average across an overlap, fuse the UAS bare ground with the lidar DEM using `r.patch.smooth` with a fixed and a spatially variable overlap, and simulate overland flow on three of the surfaces to compare the flow patterns at the edges. The optional last part animates a simulation.

Keep in mind what each surface represents. The lidar DEM is bare ground; the lidar DSM and the UAS DSM include the crop, which in June stands 1 to 2 m above the soil. A seam between two surfaces a few centimeters apart is invisible in an elevation map and obvious in shaded relief and in simulated flow.

This notebook runs the same steps as the [assignment page](https://ncsu-geoforall-lab.github.io/gis-584-uas-course/course/topics/topic_5_advanced_analytics/assignments/assignment_5a.html) in Python. You can run it in Google Colab (badge above) or locally with GRASS 8.5 installed. The GUI Profile Tool and the Animation Tool described on the assignment page are replaced here by a plotted profile and a series map.

## Learning objectives

- Map and measure the differences between a lidar DSM and UAS DSMs, and read the pattern as shift, distortion, or change
- Patch two rasters and remove the seam with a distance-weighted average across an overlap
- Fuse UAS bare ground with a lidar DEM using a fixed and a spatially variable overlap
- Simulate overland flow with path sampling and use it to judge a fused surface

## Outline

- Part 1: Environment setup
- Part 2: Compare the lidar and UAS DSMs
- Part 3: Patch the UAS DSM into the lidar DSM
- Part 4: Fuse UAS bare ground with the lidar DEM
- Part 5: Compare the overland flow on the three surfaces
- Part 6 (optional): Animate the flow
- Report questions

---
## Part 1: Environment setup

### Install GRASS 8.5

This notebook uses the `grass.tools` Python API introduced in GRASS 8.5, so we install GRASS 8.5 from the conda-forge channel using [condacolab](https://github.com/conda-incubator/condacolab).

> **Important: the next cell restarts the Colab runtime.** After you run it, Colab will report "Your session crashed for an unknown reason". This is expected and happens exactly once. Wait a few seconds for the runtime to reconnect, then continue with the next cell. If you use "Run all", run it again after the restart: the install cell detects the existing conda installation and returns immediately.

If you are running locally with GRASS 8.5 already installed, skip the three install cells.

In [ ]:
!pip install -q condacolab
import condacolab

condacolab.install()

After the runtime restarts, verify that conda is ready.

In [ ]:
import condacolab

condacolab.check()

Install GRASS 8.5 from conda-forge with mamba. This takes a few minutes.

In [ ]:
!mamba install -y -q grass=8.5

Check that GRASS is installed and which Python version is running.

In [ ]:
!grass --version

import sys

v = sys.version_info
print(f"We are using Python {v.major}.{v.minor}.{v.micro}")

### Import the GRASS Python packages

GRASS tells us where its Python packages live with `grass --config python_path`. We add that to `sys.path`, then import `grass.script` (the classic API), `grass.jupyter` (maps in notebooks), and `grass.tools` (every GRASS tool as a Python function, new in 8.5). We also import pandas and matplotlib for the profile.

In [ ]:
import io
import os
import subprocess
import time
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

# Ask GRASS where its Python packages are.
sys.path.append(
    subprocess.check_output(["grass", "--config", "python_path"], text=True).strip()
)

import grass.script as gs
import grass.jupyter as gj
from grass.tools import Tools

### Download the data

- The `Lake_Wheeler_NCspm` GRASS project (EPSG:3358, NC State Plane meters, about 225 MB); its `PERMANENT` mapset holds the 2015 UAS DSMs
- The 2013 lidar DEM and DSM of Mid Pines as GRASS raster packs (zip, about 90 MB)
- The color rules for the difference maps, `dif_lidar_uav.txt`

The downloads total about 320 MB and take a minute or two on Colab.

In [ ]:
DATA_DIR = Path("/content")  # change this if you run locally
DATA_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(DATA_DIR)

BUCKET = "https://storage.googleapis.com/gis-course-data/gis584"
PROJECT_URL = f"{BUCKET}/grassdata/Lake_Wheeler_NCspm.zip"
LIDAR_URL = f"{BUCKET}/lidar-data/Midpines/secref_mid_pines_spm_elev.zip"
COLORS_URL = f"{BUCKET}/grassdata/dif_lidar_uav.txt"

!curl -sSL -o Lake_Wheeler_NCspm.zip {PROJECT_URL} && unzip -q -o Lake_Wheeler_NCspm.zip
!curl -sSL -o secref_mid_pines_spm_elev.zip {LIDAR_URL} && unzip -q -o secref_mid_pines_spm_elev.zip
!curl -sSL -o dif_lidar_uav.txt {COLORS_URL}
!ls -lh {DATA_DIR}

### Start the GRASS session

We create a new mapset `fusion` inside the Lake Wheeler project for our work (the `PERMANENT` mapset stays untouched), start the session there, and create a `Tools` object. From here on, `tools.r_mapcalc(...)` runs `r.mapcalc`, `tools.g_region(...)` runs `g.region`, and so on; a flag like `-o` is passed as `flags="o"`.

Colab presets PROJ and GDAL to an older system install, which breaks reprojection with the conda GRASS, so we point them at the conda data files first.

In [ ]:
# Point PROJ and GDAL at the conda data files. Colab presets PROJ_LIB and
# GDAL_DATA to the older system install in /usr/share, which crashes the
# newer conda PROJ, so these must be overridden, not just defaulted.
proj_data = Path(sys.prefix) / "local" / "share" / "proj"
gdal_data = Path(sys.prefix) / "local" / "share" / "gdal"
if proj_data.is_dir():
    os.environ["PROJ_DATA"] = str(proj_data)
    os.environ["PROJ_LIB"] = str(proj_data)
if gdal_data.is_dir():
    os.environ["GDAL_DATA"] = str(gdal_data)

project = DATA_DIR / "Lake_Wheeler_NCspm"
mapset = project / "fusion"

# Create the mapset once; -c creates, -e exits without starting a shell.
if not mapset.is_dir():
    subprocess.run(["grass", "-c", "-e", str(mapset)], check=True)

session = gj.init(mapset)
tools = Tools(session=session)

print(tools.g_proj(flags="p").text)

### Helpers

`describe` prints the statistics of a raster in the current region from `r.univar`. `relief` computes a shaded relief with the vertical exaggeration the assignment uses, and `show_relief` draws one or more rasters side by side in the current region.

In [ ]:
def describe(raster, label=None):
    """Print the range, mean, and spread of a raster in the current region."""
    s = tools.r_univar(map=raster, flags="e", format="json").json
    print(f"{label or raster}: {s['n']:,} cells")
    print(f"  min {s['min']:8.2f}  max {s['max']:8.2f}  mean {s['mean']:8.2f}  median {s['median']:8.2f}  stddev {s['stddev']:6.2f}")
    return s


def relief(raster, zscale=5):
    """Shaded relief of a raster, exaggerated so that small steps cast shadows."""
    tools.r_relief(input=raster, output=f"{raster}_relief", zscale=zscale, overwrite=True)
    return f"{raster}_relief"


def show(rasters, titles=None, width=420, height=420, legend=None):
    """Draw rasters in the current region, one map each, with an optional legend."""
    titles = titles or rasters
    for raster, title in zip(rasters, titles):
        m = gj.Map(width=width, height=height, use_region=True)
        m.d_rast(map=raster)
        if legend:
            m.d_legend(raster=raster, at=(5, 45, 3, 6), title=legend, flags="sb", bgcolor="white")
        m.d_barscale(at=(55, 5))
        m.d_text(text=title, at=(3, 94), size=4, color="black", bgcolor="white")
        m.show()

---
## Part 2: Compare the lidar and UAS DSMs

### Restore the lidar surfaces

The 2013 lidar DEM (bare ground) and DSM (first surface) of Mid Pines come as GRASS raster packs. `-o` overrides the projection check, because the packs were made with an older CRS description of the same State Plane system.

In [ ]:
tools.r_unpack(input="mid_pines_lidar2013_dsm.pack", flags="o", overwrite=True)
tools.r_unpack(input="mid_pines_lidar2013_dem.pack", flags="o", overwrite=True)
print(tools.g_list(type="raster", mapset=".").text)

### Difference maps

We set the region to the lidar DSM and subtract the June 2015 UAS DSMs from it, first the one from Agisoft Metashape, then the one from Pix4D. Both were processed with the same eleven GCPs. The color rules in `dif_lidar_uav.txt` keep differences within 10 cm grey, so that only real differences show color: orange and red where the UAS surface is higher than the lidar, cyan and blue where it is lower.

In [ ]:
tools.g_region(raster="mid_pines_lidar2013_dsm", flags="p")

tools.r_mapcalc(expression="diff_lidardsm_agi_june = mid_pines_lidar2013_dsm - 2015_06_20_DSM_agi_11GCP", overwrite=True)
tools.r_mapcalc(expression="diff_lidardsm_pix4d_june = mid_pines_lidar2013_dsm - 2015_06_20_pix4d_11GCP_dsm", overwrite=True)

for raster in ["diff_lidardsm_agi_june", "diff_lidardsm_pix4d_june"]:
    tools.r_colors(map=raster, rules="dif_lidar_uav.txt")
    _ = describe(raster)

#### Map: lidar DSM minus the two UAS DSMs, drawn over the lidar shaded relief

In [ ]:
lidar_relief = relief("mid_pines_lidar2013_dsm", zscale=3)

for raster, title in [("diff_lidardsm_agi_june", "Lidar minus Agisoft, June 2015"),
                      ("diff_lidardsm_pix4d_june", "Lidar minus Pix4D, June 2015")]:
    m = gj.Map(width=520, height=520, use_region=True)
    m.d_shade(shade=lidar_relief, color=raster)
    m.d_legend(raster=raster, at=(5, 40, 3, 6), title="Difference (m)", range=(-1, 1), flags="sb", bgcolor="white")
    m.d_barscale(at=(55, 5))
    m.d_text(text=title, at=(3, 94), size=4, color="black", bgcolor="white")
    m.show()

> **Think about it:** The means are dominated by the crop, which the UAS surface includes and the 2013 lidar does not. The extremes of 30 m and more are tree crowns and building edges, where a half meter shift in position becomes a large difference in elevation. Where are the two maps alike, and where does the Pix4D map show a difference the Agisoft map does not?

### Profile along the road

Stable, hard features show systematic shifts. We sample both difference maps along a line on the road with `r.profile`, read the profiles into pandas, and summarize the differences. `r.profile -g` writes the coordinates, the distance along the line, and the raster value.

In [ ]:
ROAD = "637208,219491,637059,219734"


def profile(raster):
    text = tools.r_profile(input=raster, coordinates=ROAD, flags="g").text
    return pd.read_csv(io.StringIO(text), sep=r"\s+", header=None, names=["x", "y", "distance", raster])


road = profile("diff_lidardsm_agi_june").merge(profile("diff_lidardsm_pix4d_june"))
print(f"{len(road)} points over {road['distance'].max():.0f} m")
road[["diff_lidardsm_agi_june", "diff_lidardsm_pix4d_june"]].describe().loc[["mean", "std", "min", "max"]].round(3)

The assignment page imports the profile into a vector map with `v.in.ascii` and summarizes it with `v.univar`; the same numbers come out of pandas here. We also save the profile points as a vector so we can draw the transect on the map.

In [ ]:
road.to_csv("road.txt", sep=" ", header=False, index=False)
tools.v_in_ascii(input="road.txt", output="profile_road", separator="space",
                 columns="x double,y double,distance double,diff_agi double,diff_pix4d double", overwrite=True)
print(tools.v_univar(map="profile_road", column="diff_agi").text)

#### Plot: the three DSMs and the differences along the road

The GUI Profile Tool draws the elevation of several rasters along a transect; we do the same with matplotlib. The upper panel is the elevation of the lidar DSM and the two UAS DSMs, the lower panel the lidar minus UAS difference.

In [ ]:
elev = profile("mid_pines_lidar2013_dsm").merge(profile("2015_06_20_DSM_agi_11GCP")).merge(profile("2015_06_20_pix4d_11GCP_dsm"))

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
for col, label in [("mid_pines_lidar2013_dsm", "Lidar DSM 2013"), ("2015_06_20_DSM_agi_11GCP", "Agisoft DSM June 2015"),
                   ("2015_06_20_pix4d_11GCP_dsm", "Pix4D DSM June 2015")]:
    ax1.plot(elev["distance"], elev[col], label=label, linewidth=1)
ax1.set_ylabel("Elevation (m)")
ax1.legend()
for col, label in [("diff_lidardsm_agi_june", "Lidar minus Agisoft"), ("diff_lidardsm_pix4d_june", "Lidar minus Pix4D")]:
    ax2.plot(road["distance"], road[col], label=label, linewidth=1)
ax2.axhline(0, color="grey", linewidth=0.5)
ax2.set_xlabel("Distance along the road (m)")
ax2.set_ylabel("Difference (m)")
ax2.legend()
plt.tight_layout()
plt.show()

#### Map: the transect on the Agisoft difference map

In [ ]:
m = gj.Map(width=520, height=520, use_region=True)
m.d_shade(shade=lidar_relief, color="diff_lidardsm_agi_june")
m.d_vect(map="profile_road", color="red", icon="basic/point", size=4)
m.d_barscale(at=(55, 5))
m.show()

> **Think about it:** Why are the differences small along the road and large in the fields? Change `ROAD` to a line across a crop field and rerun the two profile cells to see what the crop does to the UAS surface.

### Clip the UAS DSM

The shift along the road is small compared with the differences in the fields, so we do not correct for it here. We use the difference maps to pick the part of the Agisoft DSM with the least distortion, clip it to that region, and copy the three surfaces to short names to keep the commands readable.

In [ ]:
tools.g_region(n=219720.9, s=219257.1, w=636762.9, e=637190.4, res=0.3, flags="p")
tools.r_mapcalc(expression="2015_06_20_DSM_agi_11GCP_cl = 2015_06_20_DSM_agi_11GCP", overwrite=True)

tools.g_copy(raster="2015_06_20_DSM_agi_11GCP_cl,uas", overwrite=True)
tools.g_copy(raster="mid_pines_lidar2013_dsm,lidar_dsm", overwrite=True)
tools.g_copy(raster="mid_pines_lidar2013_dem,lidar_dem", overwrite=True)
_ = describe("uas", "UAS DSM clip")

---
## Part 3: Patch the UAS DSM into the lidar DSM

### Simple patch

We set the region to the whole lidar DSM and fill it with the UAS DSM where the UAS has data. `r.patch` takes the first input wherever it is not null, so the UAS clip wins inside its rectangle and the lidar fills the rest. A shaded relief with `zscale=5` exaggerates the surface so that a step of a few centimeters casts a shadow.

In [ ]:
tools.g_region(raster="lidar_dsm", flags="p")
tools.r_patch(input="uas,lidar_dsm", output="lid_uas_patch", overwrite=True)
_ = relief("lid_uas_patch")

### Weighted average across an overlap

The seam is a step between two surfaces that each have a few centimeters of noise. To remove it we blend the two with a weight that goes from the lidar at the edge of the UAS clip to the UAS a few meters inside it. `r.grow.distance -n` measures the distance from the null cells inward, so `distance` is zero outside the UAS clip and grows toward its center. We blend with an overlap of 5 m and of 20 m.

In [ ]:
tools.r_grow_distance(input="uas", distance="distance", flags="n", overwrite=True)

for s in [5, 20]:
    tools.r_mapcalc(
        expression=f"patched_smooth_{s} = if(distance > {s}, uas, if(distance == 0, lidar_dsm, (1 - distance/{s}) * lidar_dsm + (distance/{s} * uas)))",
        overwrite=True,
    )
    _ = relief(f"patched_smooth_{s}")

#### Map: the seam in shaded relief

We zoom to the eastern edge of the UAS clip, where the clip boundary crosses a field, and compare the simple patch with the 5 m and 20 m blends.

In [ ]:
tools.g_region(n=219620, s=219470, w=637100, e=637250, res=0.3)
show(["lid_uas_patch_relief", "patched_smooth_5_relief", "patched_smooth_20_relief"],
     ["Simple patch", "Overlap 5 m", "Overlap 20 m"], width=400, height=400)
tools.g_region(raster="lidar_dsm")

> **Think about it:** Where exactly is the seam in the simple patch, and what happens to it at 5 m and at 20 m? What does the 20 m blend do to the UAS detail inside the clip?

---
## Part 4: Fuse UAS bare ground with the lidar DEM

### Extract the UAS bare ground

To model water flow on the UAS surface, the crop has to be removed, because water runs along the soil, not over the sorghum. The UAS cannot see the soil under the crop, so we keep only the UAS cells that lie within 30 cm of the lidar bare ground and set the rest to null.

In [ ]:
tools.g_region(raster="uas", flags="p")
tools.r_mapcalc(expression="uas_ground = if(uas - lidar_dem < 0.3, uas, null())", overwrite=True)

cells = tools.g_region(flags="g", format="json").json["cells"]
kept = describe("uas_ground", "UAS bare ground")["n"]
print(f"{kept / cells:.0%} of the clip is within 30 cm of the lidar ground")

#### Map: the UAS bare ground in yellow over the lidar relief

In [ ]:
gs.write_command("r.colors", map="uas_ground", rules="-", stdin="0% yellow\n100% yellow\n", quiet=True)

m = gj.Map(width=520, height=520, use_region=True)
m.d_rast(map=lidar_relief)
m.d_rast(map="uas_ground")
m.d_barscale(at=(55, 5))
m.show()

### Fixed overlap with `r.patch.smooth`

`r.patch.smooth` does the same distance-weighted blend as Part 3 along every edge of the bare ground patches. It is an addon, so we install it first with `g.extension`. Addons installed after the `Tools` object was created are called with `grass.script`. The first run uses a fixed overlap of 10 m.

In [ ]:
tools.g_extension(extension="r.patch.smooth")

In [ ]:
tools.g_region(raster="lidar_dsm", flags="p")
gs.run_command("r.patch.smooth", input_a="uas_ground", input_b="lidar_dem", output="lidar_uas_ground_smooth", smooth_dist=10, overwrite=True)
_ = relief("lidar_uas_ground_smooth")

### Spatially variable overlap

With the `-s` flag the overlap width follows the elevation difference between the two surfaces along the edges: wide where they disagree, narrow where they agree. `transition_angle` is the angle of the transition in degrees, and `parallel_smoothing` is the size of the window, an odd number of cells, that smooths the zone along the edge. The `overlap` output is the weight raster.

In [ ]:
gs.run_command("r.patch.smooth", input_a="uas_ground", input_b="lidar_dem", output="lidar_uas_ground_smooth2",
               overlap="overlap", transition_angle=1.3, parallel_smoothing=13, flags="s", overwrite=True)
_ = relief("lidar_uas_ground_smooth2")
_ = describe("overlap", "Overlap weight")

#### Map: the fused bare ground, fixed and variable overlap, and the overlap raster

In [ ]:
tools.g_region(raster="uas")
show(["lidar_uas_ground_smooth_relief", "lidar_uas_ground_smooth2_relief"],
     ["Fused bare ground, overlap 10 m", "Fused bare ground, variable overlap"], width=460, height=460)

tools.r_colors(map="overlap", color="viridis")
m = gj.Map(width=460, height=460, use_region=True)
m.d_rast(map="lidar_uas_ground_smooth2_relief")
m.d_rast(map="overlap")
m.d_legend(raster="overlap", at=(5, 40, 3, 6), title="Overlap weight", flags="sb", bgcolor="white")
m.d_barscale(at=(55, 5))
m.show()
tools.g_region(raster="lidar_dsm")

> **Think about it:** Along which edges is the variable overlap wide, and what is different about the two surfaces there? Compare the two relief maps at those edges.

---
## Part 5: Compare the overland flow on the three surfaces

An overland flow simulation is a sensitive test of a fused surface, because the flow follows every step in the surface. We simulate overland flow with `r.sim.water`, a path sampling method that releases walkers as rainfall and lets them move downslope, on the simply patched DSM, on the DSM blended with a 20 m overlap, and on the fused bare ground.

### Watershed boundaries

First we look at the watersheds on the lidar DEM to pick a region that drains within the study area.

In [ ]:
tools.g_region(raster="lidar_dem", flags="p")
tools.r_watershed(elevation="lidar_dem", threshold=50000, accumulation="flowacc", basin="basin", overwrite=True)
tools.r_to_vect(input="basin", output="basin", type="area", flags="t", overwrite=True)

m = gj.Map(width=520, height=520, use_region=True)
m.d_rast(map=lidar_relief)
m.d_vect(map="basin", type="boundary", color="red", width=2)
m.d_barscale(at=(55, 5))
m.show()

### Simulate the flow

The simulation needs the first derivatives of the surface, from `r.slope.aspect`. The region reaches west of the UAS clip on purpose, so that the seam is inside it. Each run simulates 20 minutes of rainfall at 30 mm per hour; with `-t` it writes a depth and a discharge map every simulated minute, named `<name>.01` to `<name>.20`. Each run takes 5 to 10 minutes on one core, so the cell asks for two threads with `nprocs=2` (Colab has two) and prints the time.

In [ ]:
tools.g_region(n=219728.7, s=219435, w=636708.9, e=637122.6, res=0.3, flags="pa")

SURFACES = {
    "lid_uas_patch": "lid_uas_patch",
    "patched_smooth_20": "patched_smooth_20",
    "lidar_uas_ground_smooth": "ground",
}

for surface, prefix in SURFACES.items():
    start = time.time()
    tools.r_slope_aspect(elevation=surface, dx=f"{surface}_dx", dy=f"{surface}_dy", overwrite=True)
    tools.r_sim_water(elevation=surface, dx=f"{surface}_dx", dy=f"{surface}_dy", rain_value=30, man_value=0.15,
                      depth=f"{prefix}_depth", discharge=f"{prefix}_disch", nwalkers=100000, niterations=20,
                      output_step=1, hmax=0.2, halpha=8.0, hbeta=1.0, nprocs=2, flags="t", overwrite=True)
    print(f"{surface}: {time.time() - start:.0f} s")

#### Map: water depth after 20 minutes on the three surfaces

The depth is draped over the shaded relief of each surface with `d.shade`, so the flow reads against the terrain. All three maps share one color table, from white under a centimeter to dark blue at 20 cm and more.

In [ ]:
DEPTH_RULES = "0 255:255:255\n0.01 200:230:255\n0.05 80:160:255\n0.1 0:80:220\n0.2 0:0:120\n1 0:0:60\n"

depths = {surface: f"{prefix}_depth.20" for surface, prefix in SURFACES.items()}
for surface, depth in depths.items():
    gs.write_command("r.colors", map=depth, rules="-", stdin=DEPTH_RULES, quiet=True)
    _ = describe(depth)

titles = {"lid_uas_patch": "Simple patch", "patched_smooth_20": "Blend, overlap 20 m", "lidar_uas_ground_smooth": "Fused bare ground"}
for surface, depth in depths.items():
    m = gj.Map(width=520, height=520, use_region=True)
    m.d_shade(shade=f"{surface}_relief", color=depth)
    m.d_legend(raster=depth, at=(5, 40, 3, 6), title="Depth (m)", flags="sb", bgcolor="white")
    m.d_barscale(at=(55, 5))
    m.d_text(text=titles[surface], at=(3, 94), size=4, color="black", bgcolor="white")
    m.show()

#### Map: the flow at the western edge of the UAS clip

In [ ]:
tools.g_region(n=219650, s=219480, w=636720, e=636890, res=0.3)
for surface, depth in depths.items():
    m = gj.Map(width=400, height=400, use_region=True)
    m.d_shade(shade=f"{surface}_relief", color=depth)
    m.d_text(text=titles[surface], at=(3, 94), size=4, color="black", bgcolor="white")
    m.show()
tools.g_region(n=219728.7, s=219435, w=636708.9, e=637122.6, res=0.3, flags="a")

> **Think about it:** Which blue lines follow the straight edge of the UAS clip, and on which surface do they disappear? Where does the water go on the fused bare ground that it could not go on the two DSMs?

---
## Part 6 (optional): Animate the flow

The GUI Animation Tool builds an animation from the twenty depth maps. In the notebook `gj.SeriesMap` does the same: it draws the maps one after the other with a slider and can save an animated GIF. Change `prefix` to animate another simulation.

In [ ]:
prefix = "ground"
steps = [f"{prefix}_depth.{i:02d}" for i in range(1, 21)]
for step in steps:
    gs.write_command("r.colors", map=step, rules="-", stdin=DEPTH_RULES, quiet=True)

os.environ["GRASS_VERBOSE"] = "0"  # keep the rendering progress messages out of the output
series = gj.SeriesMap(width=520, height=520, use_region=True)
series.add_rasters(steps)
series.d_legend(raster=steps[-1], at=(5, 40, 3, 6), title="Depth (m)", flags="sb", bgcolor="white")
series.d_barscale(at=(55, 5))
series.show()
os.environ.pop("GRASS_VERBOSE")

In [ ]:
series.save(f"{prefix}_flow.gif", duration=300)
print(f"saved {DATA_DIR / (prefix + '_flow.gif')}")

---
## Report questions

Answer each question in the Markdown cell below it. These answers, with the maps from this notebook, are the content of your report (see the [assignment page](https://ncsu-geoforall-lab.github.io/gis-584-uas-course/course/topics/topic_5_advanced_analytics/assignments/assignment_5a.html) for the submission details).

#### Question 1: Difference maps and profile

Describe the two difference maps and their statistics, and the road profile statistics for both UAS DSMs. What do the patterns show: a systematic shift, a processing distortion, or real change? Which UAS DSM would you fuse with the lidar, and why?

`Add text here with your answer to question 1`

#### Question 2: Patching

Compare the shaded relief of the simple patch and of the 5 m and 20 m blends at the clip edge. Explain how the weighted average works and what the overlap width changes.

`Add text here with your answer to question 2`

#### Question 3: Fused bare ground

Explain the bare ground test, compare the two `r.patch.smooth` results, and describe where the variable overlap is wide and why.

`Add text here with your answer to question 3`

#### Question 4: Flow comparison

Compare the depth maps after 20 minutes for the three surfaces, at the seams and in the fields. Which patterns are artifacts of the surface and which are real drainage?

`Add text here with your answer to question 4`

#### Question 5 (optional): Animation

What does the animation show that the final depth map does not?

`Add text here with your answer to question 5`